In [ ]:
import sys
import os
sys.path.append(os.path.realpath('./src/'))
import numpy as np
import torch
import matplotlib.pyplot as plt
import time

torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=20,linewidth=100000000000000)
np.set_printoptions(precision=20,suppress=True,linewidth=np.inf)
from src.utilFuncs import *
from src.optimizeFrame3 import OptimizeFrame
from src.mesher import MeshFrame

In [ ]:


####################################
# check readme
####################################
params = {'base': 2.,'height':5.,'phi':0.0,'delta':0.0, 'nx': 1, 'ny':1, 'nz':1,
            'Name':'C4','Shape':'Square'}

scale = 25.4 # inch to mm 
H = 0.5*scale# 0.5 inch per row height

W = 1*scale  # 1 inch
B = 1*scale  # 1 inch

params['height'] =  H
params['base'] =  H

params['Name']='C4'
p = 2

mf = MeshFrame()

nodeXY,connectivity,radiiElemIndex = mf.generateCombined3DLattice(params)
# radiiNodIndex,radiiElemIndex help to map radii of one unit cell to multiple unit cell
numUnitLatElem = max(radiiElemIndex) + 1        
#calculate the total length of the beam elements
lengthElem = np.linalg.norm(nodeXY[connectivity[:,0],:] - nodeXY[connectivity[:,1],:],axis=1)
totalLength = np.sum(lengthElem)
print("Total length of the beams in mm = ", totalLength)

E = torch.tensor([17.0])
elemSize = 0.8 #0.8

midpointsZ = (nodeXY[connectivity[:,0],-1] 
             + nodeXY[connectivity[:,1],-1]) / 2
midpointsZ = np.round(midpointsZ,decimals=3)
midpointsUnique = np.unique(midpointsZ)

alpha = []
if p == 2:
    alphaTop = (midpointsZ>max(nodeXY[:,-1])/2)*1.0 + (midpointsZ == max(nodeXY[:,-1])/2)*0.5
    alphaBot = (midpointsZ<max(nodeXY[:,-1])/2)*1.0 + (midpointsZ == max(nodeXY[:,-1])/2)*0.5
    alpha.append(alphaBot);
    alpha.append(alphaTop)
else:       
    if len(midpointsUnique) > p:
        listComb = [1,2,2,3,2,2,1] # group few of the beams together to apply same radius
        count = 0;
        for i in range(len(listComb)):
            if listComb[i] == 1:
                mat = 1.0*(midpointsZ == midpointsUnique[count]) 
                count +=1
            elif listComb[i] == 2:
                mat1 = 1.0*(midpointsZ == midpointsUnique[count]) 
                mat2 = 1.0*(midpointsZ == midpointsUnique[count+1])
                mat = mat1 + mat2
                count +=2
            elif listComb[i] == 3:
                mat1 = 1.0*(midpointsZ == midpointsUnique[count]) 
                mat2 = 1.0*(midpointsZ == midpointsUnique[count+1])
                mat3 = 1.0*(midpointsZ == midpointsUnique[count+2])
                mat = mat1 + mat2 + mat3
                count += 3 
            alpha.append(mat)
    else:
        for i in range(len(midpointsUnique)):
            alpha.append(1.0*(midpointsZ == midpointsUnique[i]))
        
alpha = np.stack(alpha,axis=0)
r0 = np.ones(p)*0.5 # 0.5 radius in mm
# r0 = np.array([0.7,0.7])
# r0 = np.array([0.25,0.75])
# r0 = np.linspace(0.2,1.,p)
R = np.einsum('ij,i->j',alpha,r0)

cleanImage = True

fig_set = plt.figure(figsize=(10,10))
add_title = str(p)+' unique radius '
add_title = ''
result = plotStructure(np.round(R,2),nodeXY,connectivity,add_title,plotDeformed = False,TrueScale=True,fig=fig_set,
    thicknessPlot=True,elemAnnotate=False,nodeAnnotate=False)

# Handle return value: could be ax alone or (ax, cbar)
if isinstance(result, tuple):
    ax, cbar = result
    cbar.ax.set_position([0.65, 0.3, 0.8, 0.4])
else:
    ax = result
    
# cbar.remove()
# keep z limit first
ax.set_zlim(0, H)
# get current x,y limits (may require a draw if axes auto-updates)
x0, x1 = ax.get_xlim()
y0, y1 = ax.get_ylim()
z0, z1 = ax.get_zlim()

xspan = abs(x1 - x0)
yspan = abs(y1 - y0)
zspan = abs(z1 - z0)

# set relative box aspect so physical scale is equal
ax.set_box_aspect((xspan, yspan, zspan))
# fig_set.savefig('Images/New/StructureC4_2_3x3.png', bbox_inches='tight', pad_inches=0, dpi=300,transparent=True)

if cleanImage:
    # cbar.remove()
    # remove everything in the background from ax, just plot, no axes, no ticks, no labels, no spines, no title, no bars
    ax.set_axis_off()
    ax.set_title('')
    # fig_set.savefig('Images/New/StructureC4_2_1x1.png', bbox_inches='tight', pad_inches=0, dpi=300,transparent=True)


In [ ]:

inputCurve = (np.array([0.0000000000000000, 0.6350000000000000, 1.2700000000000000,
1.9050000000000000, 2.5400000000000000, 3.1749999999999998,
3.8099999999999996, 4.4449999999999994, 5.0799999999999992,
5.7149999999999990, 6.3499999999999988])/1.25, 
8.0*np.array([0.0000000000000000, 1,1,1,1,1,1,1,1,1,1])) # 

max_x = 1.0
min_x = 0.2
ObjType = "FD"   
        
varSetup = {'min_x':min_x,'max_x':max_x}
path = params['Name']+'NN'+str(p)+'var/' # to save the results and data

# # Represent the current curve as a function of normalized displacement
# ui = torch.linspace(inputCurve[0].min(), inputCurve[0].max(), steps=100, dtype=torch.float64)
# t = (ui - ui.min()) / (ui.max() - ui.min())
# Fi = 5 * (ui)**(1/2.5)
# optFrame = OptimizeFrame(None,inputCurve)
# fig = optFrame.plot_fd_curves(
#             ui,
#             Fi,
#             label_tag="Current FD Curve",
#             target_only=False,
#             fig=plt.figure(figsize=(8, 6)),
#             normalize=False,
#             save_dir='Images/New/Target_Current_FD.png',
#             )


In [ ]:

########################################################
frameNN = torch.load(surrogateModelLoc)# switch device to cpu if needed,map_location=torch.device('cpu')) # Load to device
frameNN.eval()  # Set to evaluation mode
optFrame = OptimizeFrame(frameNN,inputCurve)
inputdim = optFrame.frameNN.nnSettings['inputDim']

fileSavePath = path+'NNmultiStart' # to save the figures of optimization run
# Check if the savePath exists
if not os.path.exists(fileSavePath):
    # Create the directory
    os.makedirs(fileSavePath)
    
st = time.perf_counter()
Algo = 'MMA'# 'L-BFGS-B', 'SLSQP','TNC'
seedOptNum = 2 #2 for C4, and 3 for C12 
np.random.seed(seedOptNum)
print("seed number for optimization = ", seedOptNum)
x0_all = np.random.rand(5,inputdim)
xopt_all = np.zeros_like(x0_all)
for i in range(x0_all.shape[0]):
    print(f"x0 = {x0_all[i,:]}, for start point = {i+1}")
    x,fun=optFrame.optimizerRun(x0_all[i,:],algorithm=Algo,tol=1e-10,fileSaveLoc=fileSavePath)
    print("Opt x = ",x)
    xopt_all[i,:] = x
    print("##################################################")
    
Total_opt_time = time.perf_counter()-st
print("Time for optimization in sec = ", Total_opt_time)   
print("##########################################")

###################
output_filename = 'DataVar/'+params['Name']+'solution_'+str(p)+'var_seedNum0.txt'

with open(output_filename, 'w') as f:
    f.write(f"# Seed: {0}, Variables: {p}, Samples: {10}\n")
    f.write(f"# Each row represents a sample in the format [val1 ... valp]\n")

    # Write the samples data row by row in the desired format
    for sample_row in np.concatenate((x0_all,xopt_all),axis=0):
        # Format each float value to a specific number of decimal places and join with a space
        # Then wrap the entire string in square brackets
        formatted_row = " ".join([f"{val:.16f}" for val in sample_row]) # Using .16f for high precision
        f.write(f"[{formatted_row}]\n")
print("Initial guesses and optimization results saved to file:",output_filename)
print(f" Run abaqus on these {output_filename} for validation")
###################


In [ ]:

print("##########################################")
solutionfilename = path + 'abaqusComp40'+params['Name']+'solution_'+str(p)+'var_seedNum0.txt'
if not os.path.exists(solutionfilename):
    print("Run abaqus on result file:",output_filename)
    sys.exit()
else:    
    print(f"Abaqus analysis file {solutionfilename} already exists, using it. Delete this if its an older file")
    #### read the abaqus solution curves
    R, U, F, analysisState = extract_ABQ_values(solutionfilename,torch.from_numpy(inputCurve[0]).reshape(-1)) # for abaqus code results
    DataAbaqusOpt = np.stack((U[5::,:],F[5::,:]),axis=2)
    analysisState = analysisState[5::]
            
    V0   = np.zeros_like(x0_all[:,0])
    Vopt = np.zeros_like(x0_all[:,0])
        
    V0, Vopt,VDataAbaqusOpt = plot_optimization_curves(optFrame, x0_all, xopt_all, V0, Vopt, DataAbaqusOpt)
    plt.savefig(path+'/Opt'+str(inputdim)+'var.png', bbox_inches='tight') # 'tight' removes extra whitespace
    
    print("########")
    print("x0_all,xopt_all = ",x0_all,xopt_all)
    print("V0,Vopt,VDataAbaqusOpt  = ",V0,Vopt,VDataAbaqusOpt)
    print("########")
    
    VDataAbaqusOpt[analysisState==0.0] = 0.0
    barPlotofOptimization(V0,Vopt,VDataAbaqusOpt)
    plt.savefig(path+'/BarOpt'+str(inputdim)+'var.png', bbox_inches='tight') # 'tight' removes extra whitespace
    
    if inputdim==2:
        xGlobalNN,ax =plot_objective_landscape(optFrame, 50)
    
        ax = plotOptxk(ax,optFrame,x0_all,"Initial Guesses ",'red','o',markersize=100)
        ax = plotOptxk(ax,optFrame,xopt_all,"Optimized Results ",'aqua' ,'^',markersize=100)
        
        ax = plotOptxk(ax,optFrame,xGlobalNN,"Gridsearch Minima ",'lime' ,'h',markersize=200)
        
        ax.legend(loc='lower right',fontsize=16,bbox_to_anchor=(0.95,0.05))
        plt.savefig(path+'/ObjSpaceNNFullData.png', bbox_inches='tight') # 'tight' removes extra whitespace


In [ ]:
pause_savefig = True

print("##########################################")
solutionfilename = path + 'abaqusComp40'+params['Name']+'solution_'+str(p)+'var_seedNum0.txt'
if not os.path.exists(solutionfilename):
    print("Run abaqus on result file:",output_filename)
    sys.exit()
else:    
    print(f"Abaqus analysis file {solutionfilename} already exists, using it. Delete this if its an older file")
    #### read the abaqus solution curves
    R, U, F, analysisState = extract_ABQ_values(solutionfilename,torch.from_numpy(inputCurve[0]).reshape(-1)) # for abaqus code results   
    DataAbaqusOpt = np.stack((U[5:10,:],F[5:10,:]),axis=2)
    analysisState = analysisState[5:10]
    x0_all = R[0:5,:]
    xopt_all = R[5:10,:]
    # print(xopt_all.shape)
    # x0_all[-1,:] = 0.8 * x0_all[-1,:]
    V0   = np.zeros_like(x0_all[:,0])
    Vopt = np.zeros_like(x0_all[:,0])
        
    V0, Vopt,VDataAbaqusOpt = plot_optimization_curves(optFrame, x0_all, xopt_all, V0, Vopt, DataAbaqusOpt)
    if not pause_savefig:
        plt.savefig(path+'/Opt'+str(inputdim)+'var.png', bbox_inches='tight',transparent=True) # 'tight' removes extra whitespace
    
    print("########")
    print("x0_all,xopt_all = ",x0_all,xopt_all)
    print("V0,Vopt,VDataAbaqusOpt  = ",V0,Vopt,VDataAbaqusOpt)
    print("########")
    
    VDataAbaqusOpt[analysisState==0.0] = 0.0
    barPlotofOptimization(V0,Vopt,VDataAbaqusOpt,threshold=4.5)
    if not pause_savefig:
        plt.savefig(path+'/BarOpt'+str(inputdim)+'var.png', bbox_inches='tight',transparent=True) # 'tight' removes extra whitespace
    
    if inputdim==2:
        xGlobalNN,ax =plot_objective_landscape(optFrame, 50, fig=plt.figure(figsize=(12,12)))
    
        ax = plotOptxk(ax,optFrame,x0_all,"Initial Guesses ",'red','o',markersize=100)
        ax = plotOptxk(ax,optFrame,xopt_all,"Optimized Results ",'aqua' ,'^',markersize=100)
        
        ax = plotOptxk(ax,optFrame,xGlobalNN,"Gridsearch Minima ",'lime' ,'h',markersize=200)
        
        ax.legend(loc='lower right',fontsize=16,bbox_to_anchor=(0.95,0.05))
        if not pause_savefig:
            plt.savefig(path+'/ObjSpaceNNFullData.png', bbox_inches='tight',transparent=True) # 'tight' removes extra whitespace
